# Lab 1 - LogPrécis: does it actually work?

In the lecture we ran LogPrécis on a handful of sessions and it looked convincing. Looking convincing is not a result.

 In this notebook you **measure** it, on the 72
sessions the authors held out of training.

You will not train anything. The model is
[`SmartDataPolito/logprecis`](https://huggingface.co/SmartDataPolito/logprecis) - a CodeBERT domain-adapted on >20k Unix sessions and
fine-tuned for token classification on 360 labelled ones.

**What you build, in order**

1. Word-level alignment - the model labels *tokens*, the analyst reads *words*
2. A context ablation - proof that the neighbours decide the tactic
3. A real evaluation - accuracy and a confusion matrix on held-out data
4. The 512-token cliff - find the bug, fix it, watch the score jump

Cells marked `# TODO` are yours. Everything else is scaffolding.

*⏰ Budget: about an hour.*

In [ ]:
# Setup - identical in every lab notebook; works on Colab and on a local clone.
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:
    subprocess.run("pip -q install transformers torch pandas pyarrow scikit-learn matplotlib".split(), check=True)
    if not Path("Lecture-LLM_Cybersecurity").exists():
        subprocess.run("git clone -q --depth 1 https://github.com/MatteoBoffa/Lecture-LLM_Cybersecurity.git".split(), check=True)
    os.chdir("Lecture-LLM_Cybersecurity")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "logprecis_lab.py").exists())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

import logging, warnings                      # the Hub is chatty; we are not debugging it
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

import labs_lib
labs_lib.style()
print("ready -", ROOT)

## Part 1 - One session, word by word

Load the model. The first call downloads ~500 MB and caches it.

In [ ]:
from transformers import pipeline

logprecis = pipeline("token-classification", model="SmartDataPolito/logprecis")
print("Those are the tactics the model is trained to recognize:", labs_lib.TACTICS)

The lecture's opening example - one line of shell, two intents:

In [ ]:
from logprecis_lab import SESSION

SESSION

In [ ]:
print("Those are the predictions the model makes on the session:")
raw = logprecis(SESSION)
raw[:4]

Look at what came back.

 Each entry is a **sub-word token** with a character span - `entity`, `start`, `end`. 

The word `/etc/init.d/iptables` is several tokens. Nothing in that list tells you where a word begins.

**Turning this into something an analyst can read is your first job.**

The convention in named-entity recognition is simple: **a word takes the label of its first token**.

In [ ]:
def word_tactics(session, predictions):
    """One (word, tactic) pair per whitespace-separated word of `session`."""
    # TODO: walk the words of `session` left to right, tracking a cursor into the string
    # TODO: `session.index(word, cursor)` gives you where this occurrence starts
    # TODO: a prediction belongs to the word when `start` falls inside the word's span
    # TODO: keep the FIRST such prediction; if there is none, fall back to "Other"
    raise NotImplementedError("your turn")


pairs = word_tactics(SESSION, raw)
width = max(len(w) for w, _ in pairs)
for word, tactic in pairs:
    print(f"{word:<{width}}  ->  {tactic}")

The firewall goes down (**Impact**), then the payload is fetched, made
executable and run (**Execution**).

In [ ]:
from logprecis_lab import tactics_of

assert word_tactics(SESSION, raw) == tactics_of(logprecis, SESSION)
print("matches logprecis_lab.tactics_of")

Collapse consecutive repetitions and the session becomes its **fingerprint**.

The *how* is gone, the *why* remains. You will use this heavily in Lab 2.

In [ ]:
from logprecis_lab import fingerprint

fingerprint(pairs)

## Part 2 - The neighbours decide

We claimed the same command serves different intents depending on context. 

Here are the paper's three `rm` sessions (Tab. 8):

In [ ]:
from logprecis_lab import RM_SESSIONS

for session in RM_SESSIONS:
    tactic = next(t for w, t in tactics_of(logprecis, session) if w == "rm")
    print(f"{tactic:<14}  <-  {session[:100]}")

Three tactics, one command. *Which* neighbours do the work?

Take the first session, where `rm -rf` wipes a file the attacker had just written, and feed the model **growing prefixes**.

First the statements up to `rm`; then one more, then one more. Watch when the label settles.

In [ ]:
probe = RM_SESSIONS[0]
statements = labs_lib.divide_statements(probe)
rm_at = next(i for i, s in enumerate(statements) if " rm " in f" {s} ")

print(f"`rm` is statement {rm_at} of {len(statements)}\n")


def tactic_of_rm(text):
    """The tactic the model assigns to the first `rm` in `text`, or None."""
    # TODO: run the model on `text` (the function tactics_of does the alignment for you)
    # TODO: return the tactic of the first word equal to "rm"
    raise NotImplementedError("your turn")


for k in range(rm_at + 1, len(statements) + 1):
    prefix = " ".join(statements[:k])
    print(f"+{k - rm_at - 1} statement(s) after it:  {str(tactic_of_rm(prefix)):<16} | {prefix}")

With the session cut off right after the `rm`, the model calls it **Defense
Evasion**: an attacker deleting a file is covering their tracks. 

Add the *next* statement - the `cat` that reads the file back - and the label becomes
**Discovery**: the delete was not about hiding anything, it was the middle step
of *can I even write on this machine?*

Notice: A Word2Vec embedding has one vector for `rm` and cannot express this at all.

In [ ]:
# The other two sessions are stable under the same probe. Why?
for i in (1, 2):
    statements_i = labs_lib.divide_statements(RM_SESSIONS[i])
    at = next(j for j, s in enumerate(statements_i) if " rm " in f" {s} ")
    labels = [str(tactic_of_rm(" ".join(statements_i[:k]))) for k in range(at + 1, len(statements_i) + 1)]
    print(f"session {i}: {' -> '.join(labels)}")

> **❓ Question.** Would you deploy LogPrécis on a *live* session, where the future statements have not been typed yet?

## Part 3 - Now measure it

The authors released their labelled corpus: 360 sessions, each statement assigned a MITRE tactic by experts, split 287 / 72 by their own `split_partitions.py`. 

We evaluate on the **72 held-out** ones.

In [ ]:
truth = labs_lib.ground_truth("test")
print(f"{len(truth)} sessions, {sum(len(t) for t in truth.tactics)} labelled statements\n")

print("Showing an example:")
row = truth.iloc[3]
for statement, tactic in zip(row.statements, row.tactics):
    print(f"{tactic:<16} {statement}")

**Problem**: The ground truth is one tactic **per statement**; the model gives you one per **word**. 

You need to bridge that. The obvious way, is a **majority vote** over the words of each statement.

In [ ]:
from collections import Counter


def statement_tactics(session):
    """One predicted tactic per statement of `session`."""
    # TODO: get (word, tactic) pairs for the whole session with tactics_of
    # TODO: use labs_lib.divide_statements(session), consuming as many pairs
    #       as that statement has words (the two agree, by construction)
    # TODO: the statement's tactic is the most frequent among its words;
    #       if it received no prediction at all, use "Other"
    raise NotImplementedError("your turn")

print("Originally:")
print([el[1] for el in pairs])
print("\nAfterwards:")
print(statement_tactics(SESSION))

Now use the model to **predict tactics for the held-out sessions**.

In [ ]:
# few seconds on a CPU.
predicted = [statement_tactics(s) for s in truth.session]

y_true = [t for row in truth.tactics for t in row]
y_pred = [t for row in predicted for t in row]
len(y_true), len(y_pred)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print(f"statement-level accuracy: {accuracy_score(y_true, y_pred):.3f}\n")
print(classification_report(y_true, y_pred, zero_division=0))

The paper reports around **0.91**. **You are well short of it.**

Before blaming the model, look at the report: the model is weirdly predicting `Other` - far
more often than `Other` actually occurs. 

Count when that is the case:

In [ ]:
print(f"predicted Other: {y_pred.count('Other'):>4}")
print(f"     true Other: {y_true.count('Other'):>4}")

That is not a model that is confused. 

"Other" seems to be the fallback answer for the model for words that receive **no prediction at all**.

Which raises the question: why would a word get no prediction?

## Part 4 - The 512-token cliff

BERT-family models have a maximum input length. LogPrécis inherits CodeBERT's:**512 tokens**. 

Real honeypot sessions might be longer than that!

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("SmartDataPolito/logprecis")
lengths = truth.session.map(lambda s: len(tokenizer(s)["input_ids"]))

print(f"\nmedian {int(lengths.median())} tokens, max {lengths.max()}")
print(f"over the 512 limit: {(lengths > 512).sum()} of {len(lengths)} sessions")

The `pipeline` does not raise on those. It silently stops predicting.
Measure exactly how much of each session it actually looked at.

In [ ]:
def coverage(session):
    """Fraction of `session`'s characters that received any prediction at all."""
    # TODO: run the model; the last prediction's "end" is the last character reached
    # TODO: return that as a fraction of len(session) - and handle the empty case
    raise NotImplementedError("your turn")


covered = truth.session.map(coverage)
print(f"sessions the model did not read to the end: {(covered < 0.99).sum()} of {len(covered)}")
print(f"worst case: {covered.min():.1%} of the session")

One session in seven is truncated, and the worst is cut after 30% of its
text. Every statement past the cut was labelled `Other` by our fallback - which is
where the score went.

This is the problem the paper solves with **context chunking**: cut the session
into windows of a few statements, label each window on its own, and concatenate.
Build it.

In [ ]:
def chunked_statement_tactics(session, max_statements=10):
    """As statement_tactics, but never feeding the model more than it can read."""
    # TODO: split the session into statements, then into windows of at most
    #       `max_statements` of them
    # TODO: join each window back into a string, run statement_tactics on it
    # TODO: concatenate the results - and make sure you return exactly one
    #       tactic per statement, padding with "Other" if a window comes up short
    raise NotImplementedError("your turn")


assert len(chunked_statement_tactics(truth.session.iloc[0])) == len(truth.tactics.iloc[0])
print("one tactic per statement")

In [ ]:
for window in (5, 10, 20):
    chunked = [chunked_statement_tactics(s, window) for s in truth.session]
    flat = [t for row in chunked for t in row]
    print(f"max_statements={window:>3}:  accuracy {accuracy_score(y_true, flat):.3f}")

From **0.78** to **0.92**, and the paper's number is reproduced - by fixing our harness, not the model. 

Now, keep the best setting and look at what the model still gets wrong.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

best = [t for row in (chunked_statement_tactics(s, 10) for s in truth.session) for t in row]
present = [t for t in labs_lib.TACTICS if t in set(y_true) | set(best)]

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay.from_predictions(
    y_true, best, labels=present, normalize="true", cmap="logprecis",
    colorbar=False, values_format=".2f", ax=ax, text_kw={"fontsize": 8},
)
ax.grid(False)                                # a grid on top of a heatmap is just noise
ax.set_xlabel("predicted tactic")
ax.set_ylabel("true tactic")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

> **❓ Question.** Which tactic is hardest, and does that surprise you? (Hint: Look at
> its support in the report.)
>
> **❓ Question.** We just used non-overlapping windows. The paper overlaps them,
> carrying a few statements of context across the boundary. Given what you found in
> Part 2, what's the intuition behind this choice?

## Take-home

- `logprecis_lab.tactics_of` in the lecture has the same `"Other"` fallback you
  wrote. Now you know when it lies.
- **TODO**: Build the dumbest possible baseline: a dictionary mapping commands to tactics
  (`wget` -> Execution, `uname` -> Discovery, ...), applied per statement. 

  Score it on the same 72 sessions. How much of the 0.92 do 130M parameters actually buy?
- Time both. Keep the number in mind for the next lab. Here, we made 72 predictions in seconds. Later, the agent will take minutes per incident. 

**Next:** `2_characterization.ipynb` - inference on 10 000 real honeypot sessions.